# SSL400 Sinhala Sign Language 
## I3D Kinetics-400 Feature Extraction Training

This notebook trains the custom classification head using a frozen I3D backbone.

In [ ]:
# 1. Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# 2. Change Directory
import os
os.chdir('/content/drive/MyDrive/ssl400_research_project')
print("Current Directory:", os.getcwd())

In [ ]:
# 3. Force Colab to use Native GPU TensorFlow
import os
os.environ["TF_USE_LEGACY_KERAS"] = "1"

import yaml
EXP_ID = 1   # Change this to 2, 3, 4, or 5 to test other enhancements
BATCH_SIZE = 8

with open('config.yaml') as f:
    config = yaml.safe_load(f)
print(f"Experiment {EXP_ID}: {config['experiments'][EXP_ID]['name']}")

In [ ]:
# 4. Start Training (With Auto-Resume)
import logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s [%(levelname)s] %(message)s')

import numpy as np
import random
import tensorflow as tf
print("GPU Devices Available:", tf.config.list_physical_devices('GPU'))

seed = config['project']['seed']
random.seed(seed)
np.random.seed(seed)
tf.random.set_seed(seed)

from src.models.i3d_builder import build_and_compile_phase1
from src.data.tf_dataset_builder import build_dataset
from src.training.callbacks import build_callbacks

num_classes   = config['model']['num_classes']
target_frames = config['video']['target_frames']
max_epochs = config['training']['max_epochs_phase1']

print(f'Building datasets for EXP{EXP_ID}...')
train_ds = build_dataset('data/splits/train_split.csv', EXP_ID, BATCH_SIZE,
                          num_classes, target_frames, augment=True, shuffle=True)
val_ds   = build_dataset('data/splits/val_split.csv', EXP_ID, BATCH_SIZE,
                          num_classes, target_frames, augment=False, shuffle=False)

print('Building I3D model (Frozen Backbone)...')
model = build_and_compile_phase1()

model_dir = config['experiments'][EXP_ID]['model_dir']
best_model_path = f"{model_dir}/best_model_phase1.keras"
if os.path.exists(best_model_path):
    print(f"\n🔥 FOUND EXISTING MODEL! Resuming training from {best_model_path} 🔥\n")
    model.load_weights(best_model_path)

callbacks = build_callbacks(EXP_ID, phase=1)

print(f'Starting training ({max_epochs} epochs)...')
history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=max_epochs,
    callbacks=callbacks,
    verbose=1,
)

In [ ]:
# 5. Save and Plot
model.save(f'{model_dir}/best_model.keras')
print(f'Final model saved to {model_dir}/best_model.keras')

import matplotlib.pyplot as plt
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
epochs = range(1, len(history.history['accuracy']) + 1)
axes[0].plot(epochs, history.history['accuracy'],  'b-', label='Train Accuracy')
axes[0].plot(epochs, history.history['val_accuracy'], 'r-', label='Val Accuracy')
axes[0].set_title(f'EXP{EXP_ID} - Accuracy')
axes[0].legend()
axes[1].plot(epochs, history.history['loss'],  'b-', label='Train Loss')
axes[1].plot(epochs, history.history['val_loss'], 'r-', label='Val Loss')
axes[1].set_title(f'EXP{EXP_ID} - Loss')
axes[1].legend()
plt.show()